# 06 — Figures & Tables

Generates **Figures 1-8** and **Tables 1-5** exactly as `08_FIGURE_TABLE_PLAN.md` describes, saving to `outputs/figures/` and `outputs/tables/`. All computation is imported from `src/`; cells only plot and save (`11_CODE_STRUCTURE.md`).

**All data is real: eBird EBD v1.16 for birds and ERA5/MODIS via Google Earth Engine for the environment (Fig 6 / Table 5 are real H1).**

In [1]:
# --- Setup: make src/ importable (works whether cwd is repo root or notebooks/) ---
import os, sys, datetime, platform
_root = os.getcwd()
while not os.path.exists(os.path.join(_root, "requirements.txt")) and _root != os.path.dirname(_root):
    _root = os.path.dirname(_root)
REPO_ROOT = _root
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import numpy as np, pandas as pd, scipy
import matplotlib
matplotlib.use("Agg")            # headless: write figure files, no GUI needed
import matplotlib.pyplot as plt

from src import load_and_clean, observer_effort, migration_metrics
from src import environmental_data as envmod
from src import statistics as stats_
from src import validation as V

FIG_DIR = os.path.join(REPO_ROOT, "outputs", "figures")
TAB_DIR = os.path.join(REPO_ROOT, "outputs", "tables")
os.makedirs(FIG_DIR, exist_ok=True); os.makedirs(TAB_DIR, exist_ok=True)

# All inputs are real: eBird EBD v1.16 for birds, ERA5/HOURLY + MODIS via Google
# Earth Engine for the environment (H1 temperature is real).
DATA_NOTE = "Real: eBird EBD v1.16 (IN-GJ, May 2026) + ERA5/MODIS via GEE"
STUDY_PERIOD = "2010-2025"
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
print("setup complete; REPO_ROOT =", REPO_ROOT)

setup complete; REPO_ROOT = /home/tops/Documents/BirdSense


In [2]:
# --- Run Stages 1-4 from src (no metric logic here; all imported) ---
stage1     = load_and_clean.run_stage1()
effort     = observer_effort.compute_observer_effort(stage1.clean_observations, stage1.clean_checklists)
metrics    = migration_metrics.compute_migration_metrics(stage1.clean_observations, stage1.clean_checklists)
# Real annual winter environment (ERA5/HOURLY + MODIS via GEE) is cached; load it.
_env_csv = os.path.join(REPO_ROOT, "data", "processed", "environmental_annual.csv")
if os.path.exists(_env_csv):
    annual_env = pd.read_csv(_env_csv)                       # REAL (GEE, cached)
    ENV_SOURCE = "REAL (ERA5/MODIS via GEE, cached)"
else:
    annual_env = envmod.build_annual_environmental(mock=True)  # fallback: FAKE
    ENV_SOURCE = "MOCK (no cached GEE table)"
print("metrics", metrics.shape, "| effort", effort.shape,
      "| annual_env", annual_env.shape, "| env:", ENV_SOURCE)

metrics (192, 14) | effort (192, 6) | annual_env (16, 5) | env: REAL (ERA5/MODIS via GEE, cached)


## Figures 1-8 -> `outputs/figures/`

In [3]:
# Figure 1 -- Study Area Map (Gujarat wetlands)
HOTSPOTS = [("Nal Sarovar",22.79,72.03),("Little Rann of Kutch",23.30,71.00),
            ("Khijadiya",22.52,70.15),("Thol Lake",23.13,72.40),
            ("Great Rann of Kutch",23.90,70.90),("Velavadar",22.03,72.02)]
bbox = V.GUJARAT_BBOX
obs = stage1.clean_observations
fig, ax = plt.subplots(figsize=(7.5, 6.5))
ax.add_patch(plt.Rectangle((bbox["lon_min"], bbox["lat_min"]),
    bbox["lon_max"]-bbox["lon_min"], bbox["lat_max"]-bbox["lat_min"],
    fill=False, edgecolor="gray", lw=1.5, ls="--", label="Gujarat bounding box"))
ax.scatter(pd.to_numeric(obs["LONGITUDE"]), pd.to_numeric(obs["LATITUDE"]),
           s=3, alpha=0.05, color="steelblue", label="observations")
for name, lat, lon in HOTSPOTS:
    ax.plot(lon, lat, "r^", ms=10)
    ax.annotate(name, (lon, lat), xytext=(5,4), textcoords="offset points", fontsize=8)
ax.set_xlabel("Longitude (deg E)"); ax.set_ylabel("Latitude (deg N)")
ax.set_title("Figure 1 -- Study Area: Gujarat & key wetlands\n" + DATA_NOTE, fontsize=10)
ax.legend(loc="upper right", fontsize=8)
fig.text(0.5, 0.005, f"Source: {DATA_NOTE}; study period {STUDY_PERIOD}", ha="center", fontsize=7)
fig.savefig(os.path.join(FIG_DIR, "figure_01_study_area.png"), dpi=130, bbox_inches="tight")
plt.close(fig); print("saved figure_01_study_area.png")

saved figure_01_study_area.png


In [4]:
# Figure 2 -- Observer Effort Growth (complete checklists per year)
eff = stats_.annual_complete_checklists(stage1.clean_checklists)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(eff["year"], eff["complete_checklists"], color="teal")
ax.set_xlabel("Year"); ax.set_ylabel("Complete checklists (count)")
ax.set_title("Figure 2 -- Observer Effort Growth\n" + DATA_NOTE, fontsize=10)
fig.text(0.5, -0.02, f"Source: {DATA_NOTE}; study period {STUDY_PERIOD}", ha="center", fontsize=7)
fig.savefig(os.path.join(FIG_DIR, "figure_02_observer_effort.png"), dpi=130, bbox_inches="tight")
plt.close(fig); print("saved figure_02_observer_effort.png")

saved figure_02_observer_effort.png


In [5]:
# Figure 3 -- Raw vs Confirmed Arrival (focal species)
focal3 = ["Anas acuta", "Anser anser", "Phoenicopterus roseus"]
rc = stats_.raw_vs_confirmed_arrival(metrics, focal3)
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), sharey=True)
for ax, sci in zip(axes, focal3):
    d = rc[rc["species"] == sci].sort_values("year")
    ax.plot(d["year"], d["raw_arrival_doy"], "o--", color="orange", label="raw earliest")
    ax.plot(d["year"], d["confirmed_arrival_doy"], "s-", color="navy", label="confirmed (2nd obs)")
    ax.set_title(stats_.common_name(sci), fontsize=9); ax.set_xlabel("Year")
axes[0].set_ylabel("Arrival day-of-year"); axes[0].legend(fontsize=8)
fig.suptitle("Figure 3 -- Raw vs Confirmed Arrival -- " + DATA_NOTE, fontsize=10)
fig.savefig(os.path.join(FIG_DIR, "figure_03_raw_vs_confirmed.png"), dpi=130, bbox_inches="tight")
plt.close(fig); print("saved figure_03_raw_vs_confirmed.png")

saved figure_03_raw_vs_confirmed.png


In [6]:
# Figures 4 & 5 -- Arrival / Departure trend, all species (small multiples)
def _trend_grid(metric_col, fignum, label, fname):
    series = stats_.species_series_with_trend(metrics, metric_col)
    fig, axes = plt.subplots(3, 4, figsize=(15, 9), sharex=True)
    for ax, (sci, info) in zip(axes.flat, series.items()):
        ax.scatter(info["years"], info["values"], s=16, color="navy")
        tr = info["trend"]
        if tr["slope"] is not None:
            xs = np.array([info["years"].min(), info["years"].max()])
            ax.plot(xs, tr["intercept"] + tr["slope"]*xs, color="crimson",
                    label=f"{tr['slope']:.2f} d/yr, p={tr['p_value']:.2f}")
            ax.legend(fontsize=7, loc="best")
        ax.set_title(info["common"], fontsize=8)
    for ax in axes[-1]: ax.set_xlabel("Year")
    for ax in axes[:, 0]: ax.set_ylabel(f"{label} day-of-year")
    fig.suptitle(f"Figure {fignum} -- {label} Date Trend, all species -- " + DATA_NOTE, fontsize=11)
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    fig.savefig(os.path.join(FIG_DIR, fname), dpi=120, bbox_inches="tight")
    plt.close(fig); print("saved", fname)

_trend_grid("first_arrival", 4, "Arrival", "figure_04_arrival_trend.png")
_trend_grid("last_departure", 5, "Departure", "figure_05_departure_trend.png")

saved figure_04_arrival_trend.png


saved figure_05_departure_trend.png


In [7]:
# Figure 6 -- Temperature vs Arrival Date (H1), all species
tv = stats_.temp_vs_arrival_points(metrics, annual_env, V.STUDY_SPECIES_SCIENTIFIC)
fig, axes = plt.subplots(3, 4, figsize=(15, 9))
for ax, (sci, info) in zip(axes.flat, tv.items()):
    ax.scatter(info["temp"], info["arrival_doy"], s=18, color="darkgreen")
    c = info["corr"]
    if c["r"] is not None and len(info["temp"]) >= 2:
        tr = stats_.linear_trend(info["temp"], info["arrival_doy"])
        xs = np.array([np.min(info["temp"]), np.max(info["temp"])])
        ax.plot(xs, tr["intercept"] + tr["slope"]*xs, color="crimson")
        ax.set_title(f"{info['common']}\nr={c['r']:.2f}, p={c['p_value']:.2f}", fontsize=8)
    else:
        ax.set_title(info["common"], fontsize=8)
for ax in axes[-1]: ax.set_xlabel("Winter mean temp (deg C)")
for ax in axes[:, 0]: ax.set_ylabel("Arrival day-of-year")
fig.suptitle("Figure 6 -- Temperature vs Arrival (H1) -- " + DATA_NOTE, fontsize=10)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(os.path.join(FIG_DIR, "figure_06_temp_vs_arrival.png"), dpi=120, bbox_inches="tight")
plt.close(fig); print("saved figure_06_temp_vs_arrival.png")

saved figure_06_temp_vs_arrival.png


In [8]:
# Figure 7 -- Habitat Category Comparison (H3)
hab_year, hab_trends = stats_.habitat_category_trends(metrics)
colors = {"wetland": "steelblue", "grassland_dryland": "sienna"}
fig, ax = plt.subplots(figsize=(8.5, 5))
for hab, g in hab_year.groupby("habitat"):
    g = g.sort_values("year")
    ax.plot(g["year"], g["arrival_doy"], "o-", color=colors[hab], label=hab)
    tr = hab_trends[hab]
    if tr["slope"] is not None:
        xs = np.array([g["year"].min(), g["year"].max()])
        ax.plot(xs, tr["intercept"] + tr["slope"]*xs, "--", color=colors[hab],
                label=f"{hab} trend {tr['slope']:.2f} d/yr (p={tr['p_value']:.2f})")
ax.set_xlabel("Year"); ax.set_ylabel("Mean confirmed arrival day-of-year")
ax.set_title("Figure 7 -- Habitat Category Comparison (H3)\n" + DATA_NOTE, fontsize=10)
ax.legend(fontsize=8)
fig.savefig(os.path.join(FIG_DIR, "figure_07_habitat_comparison.png"), dpi=130, bbox_inches="tight")
plt.close(fig); print("saved figure_07_habitat_comparison.png")

saved figure_07_habitat_comparison.png


In [9]:
# Figure 8 -- Geographic Centroid Shift (focal species), colored by year
focalC = ["Anas acuta", "Grus grus", "Phoenicopterus roseus"]
tracks = stats_.centroid_track(metrics, focalC)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
sc = None
for ax, sci in zip(axes, focalC):
    d = tracks[sci]
    ax.plot(d["centroid_longitude"], d["centroid_latitude"], "-", color="gray", alpha=0.3)
    sc = ax.scatter(d["centroid_longitude"], d["centroid_latitude"], c=d["year"],
                    cmap="viridis", s=45)
    ax.set_title(stats_.common_name(sci), fontsize=9); ax.set_xlabel("Longitude (deg E)")
axes[0].set_ylabel("Latitude (deg N)")
fig.colorbar(sc, ax=axes, label="Year", fraction=0.02, pad=0.02)
fig.suptitle("Figure 8 -- Geographic Centroid Shift -- " + DATA_NOTE, fontsize=10)
fig.savefig(os.path.join(FIG_DIR, "figure_08_centroid_shift.png"), dpi=120, bbox_inches="tight")
plt.close(fig); print("saved figure_08_centroid_shift.png")

saved figure_08_centroid_shift.png


## Tables 1-5 -> `outputs/tables/` (CSV + Markdown)

In [10]:
# Helper: save a table as CSV + Markdown (presentation only, no metric logic)
def df_to_md(df):
    cols = list(df.columns)
    lines = ["| " + " | ".join(map(str, cols)) + " |",
             "| " + " | ".join(["---"]*len(cols)) + " |"]
    for _, r in df.iterrows():
        lines.append("| " + " | ".join("" if pd.isna(v) else str(v) for v in r) + " |")
    return "\n".join(lines)

def save_table(df, num, name):
    base = os.path.join(TAB_DIR, f"table_{num:02d}_{name}")
    df.to_csv(base + ".csv", index=False)
    with open(base + ".md", "w") as fh:
        fh.write(f"# Table {num} -- {name.replace('_',' ')}\n\n")
        fh.write(f"_{DATA_NOTE}; study period {STUDY_PERIOD}_\n\n")
        fh.write(df_to_md(df) + "\n")
    print(f"saved table_{num:02d}_{name}.csv/.md  {df.shape}")
    return df

### Table 1 — Study Species

In [11]:
save_table(stats_.build_species_table(metrics), 1, "study_species")

saved table_01_study_species.csv/.md  (12, 5)


,common_name,scientific_name,habitat_category,analysis_type,confirmed_species_years_out_of_16
0,Northern Pintail,Anas acuta,wetland,full,16
1,Northern Shoveler,Spatula clypeata,wetland,full,16
2,Garganey,Spatula querquedula,wetland,full,16
3,Eurasian Wigeon,Mareca penelope,wetland,full,16
4,Common Pochard,Aythya ferina,wetland,full,16
5,Bar-headed Goose,Anser indicus,grassland_dryland,full,16
6,Greylag Goose,Anser anser,wetland,full,16
7,Common Crane,Grus grus,grassland_dryland,full,16
8,Demoiselle Crane,Grus virgo,grassland_dryland,full,16
9,Greater Flamingo,Phoenicopterus roseus,wetland,full,16


### Table 2 — Data Sources

In [12]:
save_table(stats_.build_data_sources_table(), 2, "data_sources")

saved table_02_data_sources.csv/.md  (3, 5)


,dataset,provider,years_covered,resolution,cost_usd
0,"eBird Basic Dataset (sampling + observations),...",Cornell Lab of Ornithology,2010-2025,checklist-level,0
1,"ERA5 Reanalysis (temperature, rainfall)",ECMWF via Google Earth Engine,2010-2025,"~25-30 km, daily",0
2,MODIS MOD13Q1 (NDVI),NASA via Google Earth Engine,2010-2025,"250 m, 16-day",0


### Table 3 — Data Quality Summary

In [13]:
save_table(stats_.build_data_quality_table(stage1), 3, "data_quality")

saved table_03_data_quality.csv/.md  (9, 4)


,stage,rule,checklists,observations
0,raw,loaded,244752,184301
1,dropped,Rule 1 (Historical type),16186,9707
2,dropped,Rule 2 (outside 2010-2025),25189,22134
3,dropped,Rule 5 (STATE != Gujarat),0,0
4,dropped,Rule 8 (coords outside bbox),13,0
5,dropped,Rule 7 (duplicate SEI),0,
6,dropped,Rule 6 (non-study species),,0
7,final,usable after cleaning,203364,152460
8,effort,effort-eligible complete checklists,178669,


### Table 4 — Migration Metrics Summary

In [14]:
save_table(stats_.build_migration_summary(metrics), 4, "migration_summary")

saved table_04_migration_summary.csv/.md  (12, 8)


,common_name,scientific_name,mean_arrival_date,arrival_slope_days_per_year,arrival_p_value,mean_departure_date,departure_slope_days_per_year,departure_p_value
0,Northern Pintail,Anas acuta,03 Jan,-0.524,0.0063,28 Dec,0.928,0.0217
1,Northern Shoveler,Spatula clypeata,02 Jan,-0.366,0.0069,28 Dec,0.919,0.0295
2,Garganey,Spatula querquedula,05 Jan,-0.974,0.0191,27 Dec,1.041,0.0144
3,Eurasian Wigeon,Mareca penelope,03 Jan,-0.715,0.0004,27 Dec,1.040,0.0102
4,Common Pochard,Aythya ferina,06 Jan,-1.194,0.0058,28 Dec,0.979,0.0148
5,Bar-headed Goose,Anser indicus,08 Jan,-1.644,0.0007,16 Nov,14.416,0.0174
6,Greylag Goose,Anser anser,04 Jan,-0.887,0.0036,27 Dec,1.050,0.0092
7,Common Crane,Grus grus,02 Jan,-0.412,0.0097,28 Dec,0.812,0.0271
8,Demoiselle Crane,Grus virgo,04 Jan,-0.868,0.0007,26 Dec,1.294,0.0011
9,Greater Flamingo,Phoenicopterus roseus,03 Jan,-0.544,0.0122,27 Dec,0.953,0.0150


### Table 5 — Correlation Results (H1)

In [15]:
save_table(stats_.build_correlation_table(metrics, annual_env), 5, "correlation_results")

saved table_05_correlation_results.csv/.md  (12, 7)


,common_name,scientific_name,n_years,pearson_r,p_value,spearman_rho,interpretation
0,Northern Pintail,Anas acuta,16,-0.424,0.1018,-0.254,none
1,Northern Shoveler,Spatula clypeata,16,-0.282,0.2894,-0.202,none
2,Garganey,Spatula querquedula,16,-0.514,0.0419,-0.310,strong
3,Eurasian Wigeon,Mareca penelope,16,-0.344,0.1925,-0.261,none
4,Common Pochard,Aythya ferina,16,-0.038,0.8889,-0.167,none
5,Bar-headed Goose,Anser indicus,16,-0.317,0.2315,-0.134,none
6,Greylag Goose,Anser anser,16,-0.276,0.3015,-0.195,none
7,Common Crane,Grus grus,16,-0.246,0.3590,-0.240,none
8,Demoiselle Crane,Grus virgo,16,-0.206,0.4445,-0.222,none
9,Greater Flamingo,Phoenicopterus roseus,16,-0.124,0.6465,0.002,none


_All figures/tables regenerate by running this notebook top to bottom on the processed data (`08_FIGURE_TABLE_PLAN.md` acceptance). All inputs are real: eBird observations + ERA5/MODIS via Google Earth Engine._